# 🍕 Phase 3: Machine Learning with Pizza Store Data

**Complete ML Course** covering:
- Supervised Learning (Classification & Regression)
- Decision Trees & Random Forest
- Gini & Entropy Math
- Ensemble Methods (Bagging, Boosting, XGBoost)
- Unsupervised Learning (K-Means)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

### 🤔 Why do we create synthetic data?

In real ML projects, you'd use actual business data. Here we create a synthetic dataset so we can **control the properties** and focus on learning the algorithms.

Notice we create a binary target `is_successful` from two conditions (sales > median AND rating > 3.5). This is a common pattern — turning business rules into a classification label.

In [ ]:
# Create Pizza Store Dataset for ML
n_stores = 200

df = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n_stores)],
    'daily_sales': np.round(np.random.normal(500, 120, n_stores)).astype(int),
    'avg_order_value': np.round(np.random.normal(18, 4, n_stores), 2),
    'customer_rating': np.round(np.random.uniform(2.5, 5.0, n_stores), 1),
    'delivery_time_min': np.round(np.random.exponential(25, n_stores) + 10, 1),
    'num_employees': np.random.randint(5, 25, n_stores),
    'store_size_sqft': np.random.choice([1000, 1500, 2000, 2500], n_stores),
    'years_open': np.random.randint(1, 15, n_stores),
})

# Create target: is_successful (1 if sales > median AND rating > 3.5)
median_sales = df['daily_sales'].median()
df['is_successful'] = ((df['daily_sales'] > median_sales) & (df['customer_rating'] > 3.5)).astype(int)

print(f'Dataset: {len(df)} stores')
print(f'Successful stores: {df["is_successful"].sum()} ({df["is_successful"].mean()*100:.1f}%)')
df.head()

---
## Part 1: Supervised Learning - Classification

### 1.1 Logistic Regression

### 🤔 Why Logistic Regression for classification?

Despite its name, logistic regression is a **classification** algorithm, not regression. It predicts the **probability** that something belongs to a class (e.g., "Will this store succeed?").

**Key steps:**
1. **Split data** into train (70%) and test (30%) — we train on one set and evaluate on another to see if the model generalizes
2. **Scale features** — logistic regression uses gradient descent, which works better when all features are on the same scale
3. **Train** — the model learns weights (coefficients) for each feature
4. **Predict** — for new data, it outputs a probability between 0 and 1

Why not just use linear regression for yes/no problems? Because linear regression can predict values like -0.3 or 1.5 — but probabilities must be between 0 and 1. The **sigmoid function** fixes this.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Prepare data
features = ['avg_order_value', 'customer_rating', 'delivery_time_min', 'num_employees']
X = df[features]
y = df['is_successful']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Successful', 'Successful']))

### 1.2 The Sigmoid Function

### 🤔 What is the Sigmoid Function and why does it matter?

The sigmoid function is the **heart of logistic regression**. It takes any number (from -∞ to +∞) and squashes it into a probability between 0 and 1.

**Formula:** σ(z) = 1 / (1 + e^(-z))

- z = 0 → probability = 0.5 (coin flip)
- z > 0 → probability > 0.5 → predict YES
- z < 0 → probability < 0.5 → predict NO

The **coefficients** tell you how each feature affects the prediction:
- Positive coefficient → higher feature value → higher probability of success
- Negative coefficient → higher feature value → lower probability of success

In [ ]:
# Sigmoid function: P(y=1) = 1 / (1 + e^(-z))
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Visualize sigmoid
z = np.linspace(-6, 6, 100)
p = sigmoid(z)

plt.figure(figsize=(10, 5))
plt.plot(z, p, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='Decision boundary (0.5)')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z = β₀ + β₁x₁ + β₂x₂ + ...')
plt.ylabel('P(y = 1)')
plt.title('Sigmoid Function: Converts any value to probability [0, 1]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Show coefficients
print("\nLogistic Regression Coefficients:")
for feat, coef in zip(features, log_reg.coef_[0]):
    print(f"  {feat}: {coef:.3f}")
print(f"  Intercept: {log_reg.intercept_[0]:.3f}")

---
## Part 2: Decision Trees & Gini/Entropy Math

### 2.1 Gini Impurity - Step by Step

### 🤔 What is Gini Impurity and why do we need it?

Decision trees need to decide **which feature to split on** at each node. Gini impurity measures how "mixed" a group is:

- **Gini = 0**: The group is pure (all same class) — perfect!
- **Gini = 0.5**: Maximum impurity for binary classification (50/50 split) — worst case

The tree picks the split that **reduces Gini the most** — it wants to create groups that are as pure as possible. Think of it like sorting laundry: you want each pile to contain only one color.

In [ ]:
# Gini Impurity: Gini = 1 - Σ(pᵢ²)

def gini_impurity(y):
    '''Calculate Gini impurity for a node.'''
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities ** 2)
    return gini

# Example calculation
print("=" * 60)
print("GINI IMPURITY CALCULATION")
print("=" * 60)

# Sample: 30 successful, 70 not successful
successful = 30
not_successful = 70
total = successful + not_successful

p_success = successful / total
p_not_success = not_successful / total

gini = 1 - (p_success**2 + p_not_success**2)

print(f"\nSample: {successful} successful, {not_successful} not successful")
print(f"\nStep 1: Calculate proportions")
print(f"  P(successful) = {successful}/{total} = {p_success:.2f}")
print(f"  P(not successful) = {not_successful}/{total} = {p_not_success:.2f}")
print(f"\nStep 2: Square each proportion")
print(f"  P(successful)² = {p_success:.2f}² = {p_success**2:.4f}")
print(f"  P(not successful)² = {p_not_success:.2f}² = {p_not_success**2:.4f}")
print(f"\nStep 3: Sum of squares")
print(f"  Σpᵢ² = {p_success**2:.4f} + {p_not_success**2:.4f} = {p_success**2 + p_not_success**2:.4f}")
print(f"\nStep 4: Gini = 1 - Σpᵢ²")
print(f"  Gini = 1 - {p_success**2 + p_not_success**2:.4f} = {gini:.4f}")
print(f"\n✓ Gini = {gini:.4f}")
print(f"  (0 = pure, 0.5 = maximum impurity for binary)")

### 2.2 Entropy - Step by Step

### 🤔 What is Entropy and how is it different from Gini?

Entropy is another way to measure impurity, borrowed from information theory. It measures the **amount of surprise** or **uncertainty** in a group:

- **Entropy = 0**: No surprise — all items are the same class
- **Entropy = 1**: Maximum surprise for binary — it's a coin flip

**Gini vs Entropy:** In practice, they give very similar results. Gini is slightly faster to compute (no logarithm). Most practitioners use Gini (sklearn's default), but it's good to understand both.

In [ ]:
# Entropy: H = -Σ(pᵢ × log₂(pᵢ))

def entropy(y):
    '''Calculate entropy for a node.'''
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    probabilities = probabilities[probabilities > 0]  # Avoid log(0)
    ent = -np.sum(probabilities * np.log2(probabilities))
    return ent

print("=" * 60)
print("ENTROPY CALCULATION")
print("=" * 60)

# Same sample
print(f"\nSample: {successful} successful, {not_successful} not successful")
print(f"\nStep 1: Calculate proportions")
print(f"  P(successful) = {p_success:.2f}")
print(f"  P(not successful) = {p_not_success:.2f}")
print(f"\nStep 2: Calculate log₂ of each")
print(f"  log₂({p_success:.2f}) = {np.log2(p_success):.4f}")
print(f"  log₂({p_not_success:.2f}) = {np.log2(p_not_success):.4f}")
print(f"\nStep 3: Multiply p × log₂(p)")
print(f"  {p_success:.2f} × {np.log2(p_success):.4f} = {p_success * np.log2(p_success):.4f}")
print(f"  {p_not_success:.2f} × {np.log2(p_not_success):.4f} = {p_not_success * np.log2(p_not_success):.4f}")
print(f"\nStep 4: Sum and negate")
ent = -(p_success * np.log2(p_success) + p_not_success * np.log2(p_not_success))
print(f"  Entropy = -({p_success * np.log2(p_success):.4f} + {p_not_success * np.log2(p_not_success):.4f})")
print(f"  Entropy = {ent:.4f}")
print(f"\n✓ Entropy = {ent:.4f}")
print(f"  (0 = pure, 1 = maximum impurity for binary)")

### 2.3 Information Gain - Choosing the Best Split

### 🤔 What is Information Gain?

Information Gain answers: **"How much did this split improve purity?"**

**Formula:** Information Gain = Parent Impurity − Weighted Average of Children's Impurity

The tree tries every possible split on every feature and picks the one with the **highest information gain**. This is how it decides:
- Which feature to split on
- What threshold to use (e.g., rating > 3.5 vs rating > 4.0)

Higher gain = better split = purer groups.

In [ ]:
# Information Gain = Parent Impurity - Weighted Child Impurity

def information_gain(parent, left, right, criterion='gini'):
    '''Calculate information gain from a split.'''
    if criterion == 'gini':
        parent_impurity = gini_impurity(parent)
        left_impurity = gini_impurity(left)
        right_impurity = gini_impurity(right)
    else:
        parent_impurity = entropy(parent)
        left_impurity = entropy(left)
        right_impurity = entropy(right)
    
    n = len(parent)
    n_left, n_right = len(left), len(right)
    
    weighted_child = (n_left/n) * left_impurity + (n_right/n) * right_impurity
    gain = parent_impurity - weighted_child
    return gain, parent_impurity, weighted_child

print("=" * 60)
print("INFORMATION GAIN - CHOOSING BEST SPLIT")
print("=" * 60)

# Parent node: 40 successful, 60 not successful
parent = [1]*40 + [0]*60

# Split A: rating > 3.5
left_a = [1]*30 + [0]*20   # 50 stores, 30 successful
right_a = [1]*10 + [0]*40  # 50 stores, 10 successful

# Split B: delivery_time < 30
left_b = [1]*35 + [0]*15   # 50 stores, 35 successful
right_b = [1]*5 + [0]*45   # 50 stores, 5 successful

gain_a, parent_gini, weighted_a = information_gain(parent, left_a, right_a)
gain_b, _, weighted_b = information_gain(parent, left_b, right_b)

print(f"\nParent Node: 40 successful, 60 not successful")
print(f"Parent Gini: {parent_gini:.4f}")

print(f"\n--- Split A: rating > 3.5 ---")
print(f"Left (rating > 3.5): 30 successful, 20 not → Gini = {gini_impurity(left_a):.4f}")
print(f"Right (rating ≤ 3.5): 10 successful, 40 not → Gini = {gini_impurity(right_a):.4f}")
print(f"Weighted Gini = (50/100)×{gini_impurity(left_a):.4f} + (50/100)×{gini_impurity(right_a):.4f} = {weighted_a:.4f}")
print(f"Information Gain = {parent_gini:.4f} - {weighted_a:.4f} = {gain_a:.4f}")

print(f"\n--- Split B: delivery_time < 30 ---")
print(f"Left (time < 30): 35 successful, 15 not → Gini = {gini_impurity(left_b):.4f}")
print(f"Right (time ≥ 30): 5 successful, 45 not → Gini = {gini_impurity(right_b):.4f}")
print(f"Weighted Gini = (50/100)×{gini_impurity(left_b):.4f} + (50/100)×{gini_impurity(right_b):.4f} = {weighted_b:.4f}")
print(f"Information Gain = {parent_gini:.4f} - {weighted_b:.4f} = {gain_b:.4f}")

print(f"\n🏆 Winner: {'Split B' if gain_b > gain_a else 'Split A'} (higher information gain)")

### 2.4 Decision Tree Classifier

### 🤔 Why limit tree depth with `max_depth`?

A decision tree with no limits will keep splitting until every leaf is pure — it **memorizes** the training data perfectly. This is **overfitting**.

Setting `max_depth=3` means the tree can only ask 3 questions deep. This forces it to find the **most important** patterns rather than memorizing noise.

**Feature importance** tells you which features the tree found most useful for splitting. Features that appear near the top of the tree (early splits) are the most important.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train decision tree
tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree.fit(X_train, y_train)

# Predictions
y_pred_tree = tree.predict(X_test)

print("Decision Tree Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tree):.2%}")

# Visualize tree
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=features, class_names=['Not Successful', 'Successful'],
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree for Store Success Prediction')
plt.tight_layout()
plt.show()

# Feature importance
print("\nFeature Importance:")
for feat, imp in sorted(zip(features, tree.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

---
## Part 3: Ensemble Methods

### 3.1 Bagging (Bootstrap Aggregating)

### 🤔 What is Bagging and why does it help?

A single decision tree is **unstable** — small changes in data can produce very different trees. Bagging (Bootstrap Aggregating) fixes this by:

1. Creating B **bootstrap samples** (random samples with replacement from training data)
2. Training one tree on each sample
3. **Averaging** their predictions (regression) or **voting** (classification)

Why does this work? Each tree sees slightly different data, so they make different mistakes. When you average many trees, the random errors **cancel out**, leaving only the true signal.

Bagging reduces **variance** (overfitting) without increasing bias.

In [ ]:
from sklearn.ensemble import BaggingClassifier

print("=" * 60)
print("BAGGING - Bootstrap Aggregating")
print("=" * 60)

# Bagging with decision trees
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=10,
    max_samples=0.8,  # 80% of data per tree
    bootstrap=True,
    random_state=42
)
bagging.fit(X_train, y_train)
y_pred_bag = bagging.predict(X_test)

print(f"\nBagging Accuracy: {accuracy_score(y_test, y_pred_bag):.2%}")
print(f"Single Tree Accuracy: {accuracy_score(y_test, y_pred_tree):.2%}")

print('''
How Bagging Works:
1. Create B bootstrap samples (sample with replacement)
2. Train one tree on each sample
3. Aggregate predictions (majority vote)

Why it works:
- Each tree sees different data → learns different patterns
- Random errors cancel when averaged
- Reduces VARIANCE without increasing bias
''')

### 3.2 Random Forest

### 🤔 How is Random Forest different from Bagging?

Random Forest = Bagging + **Random Feature Selection**

At each split, instead of considering ALL features, the tree only looks at a **random subset** (typically √n features for classification). This makes the trees even more different from each other, which improves the averaging effect.

**`max_features='sqrt'`** means: at each split, randomly pick √4 = 2 features to consider. This decorrelates the trees — if one feature is very strong, not every tree will use it first.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest = Bagging + Random Feature Selection
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    max_features='sqrt',  # √n features per split
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")

# Feature importance
print("\nFeature Importance:")
for feat, imp in sorted(zip(features, rf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

# Visualize feature importance
plt.figure(figsize=(10, 5))
importance_df = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
importance_df = importance_df.sort_values('Importance', ascending=True)
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

### 3.3 Boosting - AdaBoost

### 🤔 What is Boosting and how is it different from Bagging?

**Bagging**: Trees are trained **independently** (in parallel), then averaged.
**Boosting**: Trees are trained **sequentially** — each one fixes the mistakes of the previous one.

**AdaBoost** works by:
1. Training a weak model (shallow tree)
2. Finding which samples it got **wrong**
3. Giving those wrong samples **more weight**
4. Training the next model on the reweighted data

This way, each new model focuses on the **hardest cases**. Boosting reduces **bias** (underfitting), while bagging reduces variance.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

print("=" * 60)
print("BOOSTING - AdaBoost")
print("=" * 60)

# AdaBoost
ada = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

print(f"\nAdaBoost Accuracy: {accuracy_score(y_test, y_pred_ada):.2%}")

print('''
How AdaBoost Works:
1. Train weak learner on weighted data
2. Calculate model's error rate: ε
3. Calculate model weight: α = 0.5 × ln((1-ε)/ε)
4. Update sample weights:
   - Correct: w × e^(-α) → weight decreases
   - Wrong: w × e^(+α) → weight increases
5. Repeat with reweighted data

Why it works:
- Each model fixes previous mistakes
- Reduces BIAS (underfitting)
- Final model is strong even if each is weak
''')

# Show model weights
print("Model Weights (first 10):")
for i, weight in enumerate(ada.estimator_weights_[:10]):
    print(f"  Model {i+1}: α = {weight:.3f}")

### 3.4 Gradient Boosting

### 🤔 How is Gradient Boosting different from AdaBoost?

Both are boosting methods, but they fix mistakes differently:

- **AdaBoost**: Reweights the training samples (wrong ones get more weight)
- **Gradient Boosting**: Fits each new tree to the **residuals** (errors) of the previous model

Think of it like this: if the model predicts $400 but the actual is $500, the residual is $100. The next tree tries to predict that $100 gap. The learning rate (η) controls how much of each tree's prediction we add — smaller η = slower learning but better generalization.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

print("=" * 60)
print("GRADIENT BOOSTING")
print("=" * 60)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print(f"\nGradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb):.2%}")

print('''
How Gradient Boosting Works:
1. Start with initial prediction: F₀(x) = mean(y)
2. For each round:
   a. Calculate residuals: r = y - F(x)
   b. Fit tree to residuals
   c. Update: F(x) = F(x) + η × tree(x)

Key difference from AdaBoost:
- AdaBoost: Reweights samples
- Gradient Boosting: Fits to residuals (errors)
''')

### 3.5 Model Comparison

In [ ]:
# Compare all models (only include models that have been run)
models = {}
if 'y_pred' in dir(): models['Logistic Regression'] = accuracy_score(y_test, y_pred)
if 'y_pred_tree' in dir(): models['Decision Tree'] = accuracy_score(y_test, y_pred_tree)
if 'y_pred_bag' in dir(): models['Bagging'] = accuracy_score(y_test, y_pred_bag)
if 'y_pred_rf' in dir(): models['Random Forest'] = accuracy_score(y_test, y_pred_rf)
if 'y_pred_ada' in dir(): models['AdaBoost'] = accuracy_score(y_test, y_pred_ada)
if 'y_pred_gb' in dir(): models['Gradient Boosting'] = accuracy_score(y_test, y_pred_gb)

if not models:
    print('No models trained yet. Run the cells above first.')
else:
    print('=' * 60)
    print('MODEL COMPARISON')
    print('=' * 60)

    comparison_df = pd.DataFrame({
        'Model': list(models.keys()),
        'Accuracy': list(models.values())
    }).sort_values('Accuracy', ascending=False)

    print(comparison_df.to_string(index=False))

    # Visualize
    plt.figure(figsize=(10, 5))
    plt.barh(comparison_df['Model'], comparison_df['Accuracy'], color='steelblue')
    plt.xlabel('Accuracy')
    plt.title('Model Comparison')
    plt.xlim(0, 1)
    for i, v in enumerate(comparison_df['Accuracy']):
        plt.text(v + 0.01, i, f'{v:.2%}', va='center')
    plt.tight_layout()
    plt.show()

---
## Part 4: Unsupervised Learning - K-Means Clustering

### 🤔 What is K-Means Clustering and when do we use it?

K-Means is an **unsupervised** algorithm — it finds groups (clusters) in data **without any labels**. You don't tell it what "successful" means; it discovers natural groupings.

**How it works:**
1. Pick K random center points (centroids)
2. Assign each data point to the nearest centroid
3. Move each centroid to the center of its assigned points
4. Repeat until centroids stop moving

**The Elbow Method** helps choose K: plot inertia (within-cluster distance) vs K. The "elbow" where the curve bends is usually the best K.

**Why standardize first?** K-Means uses distance. If sales range 200–750 but rating ranges 2.5–5.0, sales would dominate the distance calculation. Standardizing puts all features on equal footing.

In [ ]:
from sklearn.cluster import KMeans

# Cluster stores based on performance metrics
cluster_features = ['daily_sales', 'customer_rating', 'delivery_time_min']
X_cluster = df[cluster_features]

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Find optimal k using elbow method
inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow
plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-cluster sum of squares)')
plt.title('Elbow Method for Optimal k')
plt.grid(True, alpha=0.3)
plt.show()

# Fit with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze clusters
print("Cluster Analysis:")
cluster_summary = df.groupby('cluster')[cluster_features].mean()
print(cluster_summary.round(2))

### 4.1 Visualize Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Sales vs Rating colored by cluster
colors = ['red', 'green', 'blue']
for cluster in range(3):
    mask = df['cluster'] == cluster
    axes[0].scatter(df.loc[mask, 'daily_sales'], 
                    df.loc[mask, 'customer_rating'],
                    c=colors[cluster], label=f'Cluster {cluster}', alpha=0.6)
axes[0].set_xlabel('Daily Sales ($)')
axes[0].set_ylabel('Customer Rating')
axes[0].set_title('Store Clusters: Sales vs Rating')
axes[0].legend()

# Cluster profiles
cluster_means = df.groupby('cluster')[cluster_features].mean()
cluster_means.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral', 'green'])
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Mean Value')
axes[1].set_title('Cluster Profiles')
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Interpret clusters
print("\nCluster Interpretation:")
for cluster in range(3):
    means = cluster_means.loc[cluster]
    print(f"\nCluster {cluster}:")
    print(f"  Sales: ${means['daily_sales']:.0f}, Rating: {means['customer_rating']:.1f}, Delivery: {means['delivery_time_min']:.0f} min")

---
## Part 5: Bias-Variance Tradeoff

### 🤔 What is the Bias-Variance Tradeoff?

This is the **most fundamental concept in ML**:

- **Bias** = how far off the model's average prediction is from the truth (underfitting)
- **Variance** = how much the model's predictions change with different training data (overfitting)

**Simple models** (shallow trees): High bias, low variance → consistently wrong
**Complex models** (deep trees): Low bias, high variance → right on training data, wrong on new data

The sweet spot is in the middle — complex enough to capture the pattern, simple enough to generalize. This is why we tune hyperparameters like `max_depth`.

In [ ]:
# Demonstrate bias-variance tradeoff with different tree depths
depths = [1, 2, 3, 5, 10, 15, None]
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

# Plot
plt.figure(figsize=(10, 5))
depth_labels = [str(d) if d else 'None' for d in depths]
x = range(len(depths))
plt.plot(x, train_scores, 'b-o', label='Train Accuracy')
plt.plot(x, test_scores, 'r-o', label='Test Accuracy')
plt.xticks(x, depth_labels)
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Bias-Variance Tradeoff: Tree Depth vs Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Bias-Variance Tradeoff:")
print("  - Shallow trees (depth=1-2): HIGH BIAS, LOW VARIANCE → Underfitting")
print("  - Deep trees (depth=None): LOW BIAS, HIGH VARIANCE → Overfitting")
print("  - Optimal: Balance between bias and variance")

---
## Summary

You've learned:
- **Classification**: Logistic regression, decision trees
- **Gini & Entropy**: Step-by-step impurity calculations
- **Ensemble Methods**: Bagging, Random Forest, AdaBoost, Gradient Boosting
- **Clustering**: K-Means with elbow method
- **Bias-Variance**: The fundamental ML tradeoff